### Setup

In [1]:
# Packages
suppressPackageStartupMessages({
  library(MASS)
  library(tidyverse)
  library(reshape2)
  library(ggplot2)
  library(cowplot)
  library(grid)
  library(vegan)
  library(emmeans)
  library(nlme)
  library(ggh4x)
})

# Paths
DATA_DIR <- "../data"
FIG_DIR  <- "../figs"

SPECIES_FILE <- file.path(DATA_DIR, "species_abundance.txt")
PHENOS_FILE  <- file.path(DATA_DIR, "init_phenotypes.txt")

### Plot constants + helpers

In [2]:
AMP_COL    <- "#6D2682"
GROWTH_COL <- "#2BBFD9"

AB_LINE_COLS <- c(
  "No"           = "#BBBBBC",
  "Low"          = "#D7C7E3",
  "Intermediate" = "#AD8CB9",
  "High"         = AMP_COL
)
AB_FILL_COLS  <- AB_LINE_COLS
AB_FILL_ALPHA <- 0.28

TRAIT_COLS <- c(
  "Comm. MIC"         = AMP_COL,
  "Comm. carrying capacity" = GROWTH_COL,
  "Shannon Index" = "black"
)

FS_ALL      <- 26
LINE_MAIN   <- 3.2
LINE_ERR    <- 1.7
VLINE_W     <- 1.1
PT_RAW      <- 3.2
BORDER_W    <- 1.2

STATE_LABELS_1LINE <- c(
  "Pre-disturbance"  = "Pre-disturbance",
  "Post-disturbance" = "Post-disturbance",
  "Post-recovery"    = "Post-recovery"
)
STATE_LABELS_B_2LINE <- c(
  "Pre-disturbance"  = "Pre-\ndisturbance",
  "Post-disturbance" = "Post-\ndisturbance",
  "Post-recovery"    = "Post-\nrecovery"
)
ABCOL_LABELS <- c(
  "No"           = "0~plain('\u00B5g')~plain(mL)^{-1}",
  "Low"          = "5~plain('\u00B5g')~plain(mL)^{-1}",
  "Intermediate" = "50~plain('\u00B5g')~plain(mL)^{-1}",
  "High"         = "500~plain('\u00B5g')~plain(mL)^{-1}"
)

ABCOL_LABELLER <- as_labeller(
  ABCOL_LABELS,
  default = label_parsed
)

theme_apex <- function() {
  theme_bw(base_size = FS_ALL) +
    theme(
      panel.grid = element_blank(),

      panel.background = element_rect(
        fill = "white",
        colour = NA
      ),

      plot.background = element_rect(
        fill = "white",
        colour = NA
      ),

      panel.border = element_rect(
        colour = "black",
        fill = NA,
        linewidth = BORDER_W
      ),

      axis.text = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      axis.title = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      # Smaller margins keep titles close to the grid
      axis.title.y = element_text(
        face = "plain",
        margin = margin(r = 8)
      ),

      axis.title.x = element_text(
        face = "plain",
        margin = margin(t = 6)
      ),

      strip.background = element_blank(),

      strip.text = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      strip.text.y = element_text(
        angle = 0,
        hjust = 0,
        face = "plain"
      ),

      strip.text.x = element_text(
        angle = 0,
        hjust = 0.5,
        face = "plain"
      ),

      legend.text = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      legend.title = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      legend.title.align = 0.5,
      legend.key.width = unit(1.6, "lines"),
      legend.spacing.x = unit(0.6, "lines"),

      plot.title = element_text(face = "plain"),
      plot.subtitle = element_text(face = "plain"),
      plot.caption = element_text(face = "plain"),

      plot.margin = margin(3, 3, 3, 3)
    )
}

range01 <- function(x) {
  x <- as.numeric(x)
  rng <- range(x, na.rm = TRUE)
  denom <- max(rng[2] - rng[1], 1e-12)
  (x - rng[1]) / denom
}

bray_curtis <- function(v1, v2) {
  v1 <- as.numeric(v1); v2 <- as.numeric(v2)
  num <- sum(abs(v1 - v2), na.rm = TRUE)
  den <- sum(v1 + v2, na.rm = TRUE)
  num / max(den, 1e-12)
}

shannon_div <- function(counts) {
  counts <- as.numeric(counts)
  tot <- sum(counts, na.rm = TRUE)
  if (!is.finite(tot) || tot <= 0) return(0)
  p <- counts / tot
  p <- p[p > 0 & is.finite(p)]
  -sum(p * log(p))
}

parse_day <- function(s) {
  dplyr::case_when(
    str_detect(s, "d4")  ~ 4,
    str_detect(s, "d8")  ~ 8,
    str_detect(s, "d12") ~ 12,
    str_detect(s, "d16") ~ 16,
    str_detect(s, "d20") ~ 20,
    TRUE ~ NA_real_
  )
}
parse_evo <- function(s) {
  dplyr::case_when(
    str_starts(s, "a_") ~ "All ancestral",
    str_starts(s, "d_") ~ "Evolved 1972",
    str_starts(s, "e_") ~ "Evolved 1977",
    str_starts(s, "f_") ~ "All evolved",
    TRUE ~ NA_character_
  )
}
parse_rep <- function(s) {
  dplyr::case_when(
    str_detect(s, "_1_") ~ 1,
    str_detect(s, "_2_") ~ 2,
    str_detect(s, "_3_") ~ 3,
    str_detect(s, "_4_") ~ 4,
    TRUE ~ NA_real_
  )
}
parse_ab <- function(s) {
  dplyr::case_when(
    str_detect(s, "_t1_") ~ 5,
    str_detect(s, "_t2_") ~ 50,
    str_detect(s, "_t3_") ~ 500,
    str_detect(s, "_t4_") ~ 0,
    TRUE ~ NA_real_
  )
}

### Load and format data

In [3]:
species_raw <- read.table(SPECIES_FILE, header = TRUE, sep = "\t", check.names = FALSE, stringsAsFactors = FALSE)
phenos_raw  <- read.table(PHENOS_FILE,  header = TRUE, sep = "\t", check.names = FALSE, stringsAsFactors = FALSE)
stopifnot("f_correct" %in% names(species_raw))

species0 <- species_raw %>%
  mutate(
    f_correct = suppressWarnings(as.numeric(f_correct)),
    Day       = parse_day(sample),
    Evo_orig  = parse_evo(sample),
    Replicate = parse_rep(sample),
    AB        = parse_ab(sample),
    AB_fac    = factor(as.character(as.integer(AB)), levels = c("0","5","50","500")),
    strainID  = str_replace_all(strainID, "-", "_")
  ) %>%
  filter(is.finite(Day), !is.na(Evo_orig), is.finite(Replicate), is.finite(AB)) %>%
  mutate(
    Replicate = factor(as.integer(Replicate)),
    Evo_fac = factor(
      Evo_orig,
      levels = c("All ancestral","Evolved 1972","Evolved 1977","All evolved"),
      labels = c("Anc. comm.","Subdom. primed","Dom. primed","Full primed")
    ),
    AB_level = factor(AB_fac, levels = c("0","5","50","500"),
                      labels = c("No","Low","Intermediate","High"))
  )

### Compute objects needed for Fig 4 and for supplementary stats

In [4]:
compute_for_species <- function(species, phenos_raw, evo_levels_keep, evo_labels_row) {

  species <- species %>%
    filter(Evo_fac %in% evo_levels_keep) %>%
    mutate(Evo_fac_row = factor(Evo_fac, levels = names(evo_labels_row), labels = unname(evo_labels_row)))

  meta <- species %>% distinct(sample, Day, Evo_orig, Evo_fac_row, AB_level, Replicate)

  species_mat <- species %>%
    group_by(sample, strainID) %>%
    summarise(f = sum(f_correct, na.rm = TRUE), .groups = "drop") %>%
    pivot_wider(names_from = strainID, values_from = f, values_fill = 0)

  mat_only  <- species_mat %>% select(-sample)
  mat_names <- names(mat_only)

  # Reference states: AB = 0 centroids at days 4,16,20
  ref_days <- c(predist = 4, postdist = 16, postrec = 20)

  ref_centroids <- meta %>%
    mutate(AB = parse_ab(sample)) %>%
    filter(AB == 0, Day %in% ref_days) %>%
    select(sample, Evo_orig, Day) %>%
    left_join(species_mat, by = "sample") %>%
    pivot_longer(cols = all_of(mat_names), names_to = "strainID", values_to = "f") %>%
    group_by(Evo_orig, Day, strainID) %>%
    summarise(centroid = mean(f, na.rm = TRUE), .groups = "drop") %>%
    mutate(State_raw = recode(as.character(Day), "4"="predist","16"="postdist","20"="postrec")) %>%
    select(Evo_orig, State_raw, strainID, centroid)

  dist_grid <- meta %>%
    left_join(species_mat, by = "sample") %>%
    tidyr::crossing(State_raw = names(ref_days))

  dist_long <- dist_grid %>%
    pivot_longer(cols = all_of(mat_names), names_to = "strainID", values_to = "f") %>%
    left_join(ref_centroids, by = c("Evo_orig","State_raw","strainID")) %>%
    group_by(sample, Day, Evo_orig, Evo_fac_row, AB_level, Replicate, State_raw) %>%
    summarise(distance = bray_curtis(f, centroid), .groups = "drop") %>%
    mutate(
      State = factor(
        recode(State_raw, predist="Pre-disturbance", postdist="Post-disturbance", postrec="Post-recovery"),
        levels = c("Pre-disturbance","Post-disturbance","Post-recovery")
      )
    )

  A_sum <- dist_long %>%
    group_by(Evo_fac_row, State, AB_level, Day) %>%
    summarise(
      mean = mean(distance, na.rm = TRUE),
      se   = sd(distance, na.rm = TRUE) / sqrt(sum(is.finite(distance))),
      .groups = "drop"
    ) %>%
    mutate(ci_low = mean - 1.96 * se, ci_high = mean + 1.96 * se)

  A_sum_line <- A_sum %>%
    group_by(Evo_fac_row, State, AB_level) %>%
    filter(sum(is.finite(mean)) >= 2) %>%
    ungroup()

  traj_metrics <- dist_long %>%
    filter(Day %in% c(4,8,12,16,20)) %>%
    arrange(Evo_orig, AB_level, Replicate, State, Day) %>%
    group_by(Evo_orig, Evo_fac_row, AB_level, Replicate, State) %>%
    summarise(
      roughness = sum(abs(diff(distance)), na.rm = TRUE),
      .groups = "drop"
    ) %>%
    ungroup()

  # Panel C data
  phenos <- phenos_raw %>%
    mutate(
      strainID = str_replace_all(strainID, "-", "_"),
      MIC = suppressWarnings(as.numeric(MIC)),
      k   = suppressWarnings(as.numeric(k))
    )

  init_anc <- phenos %>% filter(Pop == "ANC") %>% select(strainID, MIC_ANC = MIC, k_ANC = k)
  init_evo <- phenos %>% filter(Pop == "EVO") %>% select(strainID, MIC_EVO = MIC, k_EVO = k)

  init_pheno <- inner_join(init_anc, init_evo, by = "strainID") %>%
    mutate(noise = runif(n(), 1e-5, 1e-4)) %>%
    mutate(
      MIC_ANC_noise = MIC_ANC + noise,
      MIC_EVO_noise = MIC_EVO + noise,
      k_ANC_noise   = k_ANC   + noise,
      k_EVO_noise   = k_EVO   + noise
    )

  scalar_base <- species %>%
    select(strainID, f_correct, Day, Evo_orig, Evo_fac_row, AB_level, Replicate, sample) %>%
    left_join(init_pheno, by = "strainID")

  make_scalar <- function(df, evo_label, strain_special = NULL) {
    out <- df %>% filter(Evo_orig == evo_label)
    if (evo_label %in% c("All ancestral","All evolved")) {
      if (evo_label == "All ancestral") out <- out %>% mutate(MIC = MIC_ANC_noise, k = k_ANC_noise)
      if (evo_label == "All evolved")   out <- out %>% mutate(MIC = MIC_EVO_noise, k = k_EVO_noise)
    } else {
      out <- out %>% mutate(
        MIC = if_else(strainID == strain_special, MIC_EVO_noise, MIC_ANC_noise),
        k   = if_else(strainID == strain_special, k_EVO_noise,   k_ANC_noise)
      )
    }
    out
  }

  scalar_combo <- bind_rows(
    make_scalar(scalar_base, "All ancestral"),
    make_scalar(scalar_base, "All evolved"),
    make_scalar(scalar_base, "Evolved 1972", "HAMBI_1972"),
    make_scalar(scalar_base, "Evolved 1977", "HAMBI_1977")
  ) %>%
    filter(!is.na(MIC), !is.na(k)) %>%
    group_by(sample) %>%
    mutate(f_rel = f_correct / max(sum(f_correct, na.rm = TRUE), 1e-12)) %>%
    ungroup() %>%
    mutate(
      scal_prod_logMIC_i = log(MIC) * f_rel,
      scal_prod_k_i      = k * f_rel
    )

  scalar_summary <- scalar_combo %>%
    group_by(Day, Evo_orig, Evo_fac_row, AB_level, Replicate, sample) %>%
    summarise(
      scal_prod_logMIC = sum(scal_prod_logMIC_i, na.rm = TRUE),
      scal_prod_k      = sum(scal_prod_k_i, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    group_by(Evo_orig, AB_level) %>%
    mutate(
      MIC_rescaled = range01(scal_prod_logMIC),
      k_rescaled   = range01(scal_prod_k)
    ) %>%
    ungroup()

  mat2 <- species %>%
    group_by(sample, strainID) %>%
    summarise(f = sum(f_correct, na.rm = TRUE), .groups = "drop") %>%
    pivot_wider(names_from = strainID, values_from = f, values_fill = 0)

  shan <- mat2 %>%
    mutate(shannon = pmap_dbl(select(., -sample), ~ shannon_div(c(...)))) %>%
    mutate(shan_scaled = range01(shannon)) %>%
    select(sample, shan_scaled)

  scalar_plot <- scalar_summary %>% left_join(shan, by = "sample")

  scalar_long <- scalar_plot %>%
    select(sample, Day, Evo_fac_row, AB_level, MIC_rescaled, k_rescaled, shan_scaled) %>%
    pivot_longer(cols = c(MIC_rescaled, k_rescaled, shan_scaled),
                 names_to = "Trait_raw", values_to = "Value") %>%
    mutate(
      Trait = recode(Trait_raw,
                     "MIC_rescaled"="Comm. MIC",
                     "k_rescaled"="Comm. carrying capacity",
                     "shan_scaled"="Shannon Index"),
      Trait = factor(Trait, levels = c("Comm. MIC","Comm. carrying capacity","Shannon Index")),
      Linetype = if_else(Trait == "Shannon Index", "dashed", "solid")
    )

  C_sum <- scalar_long %>%
    group_by(Evo_fac_row, AB_level, Trait, Linetype, Day) %>%
    summarise(
      mean = mean(Value, na.rm = TRUE),
      se   = sd(Value, na.rm = TRUE) / sqrt(sum(is.finite(Value))),
      .groups = "drop"
    ) %>%
    mutate(ci_low = mean - 1.96 * se, ci_high = mean + 1.96 * se)

  C_sum_line <- C_sum %>%
    group_by(Evo_fac_row, AB_level, Trait, Linetype) %>%
    filter(sum(is.finite(mean)) >= 2) %>%
    ungroup()

  shade_C <- tibble(
    AB_level = factor(c("No","Low","Intermediate","High"),
                      levels = c("No","Low","Intermediate","High")),
    xmin = 4, xmax = 8, ymin = -Inf, ymax = Inf
  )

  list(
    dist_long = dist_long,
    A_sum = A_sum,
    A_sum_line = A_sum_line,
    traj_metrics = traj_metrics,
    scalar_long = scalar_long,
    C_sum = C_sum,
    C_sum_line = C_sum_line,
    shade_C = shade_C
  )
}

### Fig 4

In [23]:
# =========================================================================
# Figure 4 theme
# =========================================================================

theme_apex <- function() {
  theme_bw(base_size = FS_ALL) +
    theme(
      panel.grid = element_blank(),

      panel.background = element_rect(
        fill = "white",
        colour = NA
      ),

      plot.background = element_rect(
        fill = "white",
        colour = NA
      ),

      panel.border = element_rect(
        colour = "black",
        fill = NA,
        linewidth = BORDER_W
      ),

      # All ordinary text remains plain
      axis.text = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      axis.title = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      axis.title.x = element_text(
        face = "plain",
        margin = margin(t = 4)
      ),

      axis.title.y = element_text(
        face = "plain",
        margin = margin(r = 5)
      ),

      strip.background = element_blank(),

      strip.text = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      # Column facet labels remain horizontal
      strip.text.x = element_text(
        face = "plain",
        angle = 0,
        hjust = 0.5,
        margin = margin(
          t = 0,
          r = 0,
          b = 1,
          l = 0,
          unit = "pt"
        )
      ),

      # Row facet labels are vertical in A, B, and C
      strip.text.y = element_text(
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5,
        margin = margin(
          t = 1,
          r = 2,
          b = 1,
          l = 2,
          unit = "pt"
        )
      ),

      strip.text.y.left = element_text(
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5
      ),

      strip.text.y.right = element_text(
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5
      ),

      legend.text = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      legend.title = element_text(
        size = FS_ALL,
        face = "plain",
        colour = "black"
      ),

      legend.title.align = 0.5,

      legend.key.width = unit(
        1.35,
        "lines"
      ),

      legend.key.height = unit(
        0.70,
        "lines"
      ),

      legend.spacing.x = unit(
        0.30,
        "lines"
      ),

      legend.spacing.y = unit(
        0,
        "pt"
      ),

      legend.margin = margin(0, 0, 0, 0),
      legend.box.margin = margin(0, 0, 0, 0),

      legend.box.spacing = unit(
        0,
        "pt"
      ),

      plot.title = element_text(
        face = "plain"
      ),

      plot.subtitle = element_text(
        face = "plain"
      ),

      plot.caption = element_text(
        face = "plain"
      ),

      plot.margin = margin(
        t = 0,
        r = 0,
        b = 0,
        l = 0
      )
    )
}


# =========================================================================
# Figure 4 plotting function
# =========================================================================

make_plots <- function(
    obj,
    outfile_pdf,

    # B is slightly wider and A slightly narrower
    rel_widths_top = c(
      0.67,
      0.33
    ),

    # Relative heights of upper row and panel C
    rel_heights_main = c(
      0.48,
      0.52
    ),

    # Small blank margin before panel A
    left_pad_A = 0.006,

    width_pdf = 23.5,
    height_pdf = 18.5,

    tags = c(
      "A",
      "B",
      "C"
    ),

    grid_tick_size = max(
      FS_ALL - 5,
      8
    ),

    B_x_text_size = max(
      FS_ALL - 6,
      8
    ),

    # Slightly larger external panel letters
    panel_tag_size = 30,

    # External panel-letter positions
    tag_A_x = 0.002,
    tag_A_y = 0.845,

    tag_B_x = 0.002,
    tag_B_y = 0.990,

    tag_C_x = 0.002,
    tag_C_y = 0.985,

    # More elegant trajectory-line widths
    line_width_A = LINE_MAIN * 0.6,
    line_width_C = LINE_MAIN * 0.6
) {

  # -----------------------------------------------------------------------
  # Data
  # -----------------------------------------------------------------------

  dist_long    <- obj$dist_long
  A_sum        <- obj$A_sum
  A_sum_line   <- obj$A_sum_line
  traj_metrics <- obj$traj_metrics
  scalar_long  <- obj$scalar_long
  C_sum        <- obj$C_sum
  C_sum_line   <- obj$C_sum_line
  shade_C      <- obj$shade_C

  comp_vlines_A <- tibble(
    State = factor(
      c(
        "Pre-disturbance",
        "Post-disturbance",
        "Post-recovery"
      ),
      levels = c(
        "Pre-disturbance",
        "Post-disturbance",
        "Post-recovery"
      )
    ),

    xint = c(
      4,
      16,
      20
    )
  )


  # =======================================================================
  # Panel A
  # =======================================================================

  pA <- ggplot() +

    annotate(
      "rect",
      xmin = 4,
      xmax = 8,
      ymin = -Inf,
      ymax = Inf,
      fill = AB_FILL_COLS["Low"],
      alpha = AB_FILL_ALPHA,
      colour = NA
    ) +

    annotate(
      "rect",
      xmin = 12,
      xmax = 16,
      ymin = -Inf,
      ymax = Inf,
      fill = AMP_COL,
      alpha = 0.25,
      colour = NA
    ) +

    geom_vline(
      xintercept = c(
        4,
        8,
        12,
        16
      ),
      linetype = 2,
      linewidth = VLINE_W,
      colour = "grey35",
      alpha = 0.8
    ) +

    geom_vline(
      data = comp_vlines_A,
      aes(
        xintercept = xint
      ),
      inherit.aes = FALSE,
      linewidth = 2.4,
      colour = "black"
    ) +

    geom_point(
      data = dist_long,
      aes(
        x = Day,
        y = distance
      ),
      position = position_jitter(
        width = 0.18,
        height = 0
      ),
      alpha = 0.24,
      size = PT_RAW,
      stroke = 0,
      colour = "grey10"
    ) +

    # Thinner mean trajectories
    geom_line(
      data = A_sum_line,
      aes(
        x = Day,
        y = mean,
        colour = AB_level,
        group = AB_level
      ),
      linewidth = line_width_A
    ) +

    geom_errorbar(
      data = A_sum,
      aes(
        x = Day,
        ymin = ci_low,
        ymax = ci_high,
        colour = AB_level
      ),
      width = 0.35,
      linewidth = LINE_ERR,
      alpha = 0.95
    ) +

    ggh4x::facet_grid2(
      rows = vars(
        Evo_fac_row
      ),

      cols = vars(
        State
      ),

      axes = "all",
      remove_labels = "none",

      labeller = labeller(
        State = STATE_LABELS_1LINE
      )
    ) +

    scale_colour_manual(
      values = AB_LINE_COLS,

      breaks = names(
        ABCOL_LABELS
      ),

      labels = function(x) {
        parse(
          text = unname(
            ABCOL_LABELS[x]
          )
        )
      },

      name = "Pre-pulse priming ampicillin"
    ) +

    scale_x_continuous(
      breaks = c(
        4,
        8,
        12,
        16,
        20
      ),

      labels = c(
        "4",
        "8",
        "12",
        "16",
        "20"
      ),

      limits = c(
        3,
        21
      ),

      expand = expansion(
        mult = c(
          0.002,
          0.002
        )
      )
    ) +

    # One-decimal numerical labels
    scale_y_continuous(
      breaks = c(
        0,
        0.25,
        0.50
      ),

      labels = function(x) {
        sprintf(
          "%.1f",
          x
        )
      },

      limits = c(
        0,
        0.50
      ),

      expand = expansion(
        mult = c(
          0,
          0
        )
      )
    ) +

    coord_cartesian(
      clip = "on"
    ) +

    labs(
      x = "Day",
      y = "Bray–Curtis dissimilarity to state"
    ) +

    theme_apex() +

    theme(
      legend.position = "top",

      legend.direction = "horizontal",
      legend.box = "horizontal",
      legend.justification = "center",

      axis.text.x = element_text(
        size = grid_tick_size,
        face = "plain",
        margin = margin(t = 1)
      ),

      axis.text.y = element_text(
        size = grid_tick_size,
        face = "plain",
        margin = margin(r = 1)
      ),

      axis.title.x = element_text(
        size = FS_ALL,
        face = "plain",
        margin = margin(t = 3)
      ),

      axis.title.y = element_text(
        size = FS_ALL,
        face = "plain",
        margin = margin(r = 4)
      ),

      strip.text.y = element_text(
        size = FS_ALL,
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5
      ),

      strip.text.y.right = element_text(
        size = FS_ALL,
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5
      ),

      panel.spacing.x = unit(
        0.08,
        "lines"
      ),

      panel.spacing.y = unit(
        0.08,
        "lines"
      ),

      plot.margin = margin(
        t = 0,
        r = 0,
        b = 0,
        l = 0
      )
    ) +

    guides(
      colour = guide_legend(
        nrow = 1,
        byrow = TRUE,
        title.position = "top",

        title.theme = element_text(
          face = "plain"
        )
      )
    )


  # =======================================================================
  # Panel B
  # =======================================================================

  b_y <- expression(
    atop(
      "Temporal roughness of dissimilarity",
      paste(
        "(",
        Sigma,
        " |",
        Delta,
        " Bray–Curtis|)"
      )
    )
  )

  pB <- ggplot(
    traj_metrics,
    aes(
      x = Evo_fac_row,
      y = roughness
    )
  ) +

    geom_hline(
      yintercept = 0,
      linetype = 2,
      linewidth = 1.1,
      colour = "grey40"
    ) +

    geom_boxplot(
      width = 0.62,
      linewidth = 1.5,
      outlier.shape = NA,
      colour = "black",
      fill = "grey90"
    ) +

    geom_point(
      position = position_jitter(
        width = 0.10,
        height = 0
      ),
      alpha = 0.55,
      size = PT_RAW,
      stroke = 0
    ) +

    facet_grid(
      rows = vars(
        State
      ),

      scales = "free_y",

      labeller = labeller(
        State = STATE_LABELS_B_2LINE
      )
    ) +

    scale_y_continuous(
      labels = scales::label_number(
        accuracy = 0.1,
        trim = TRUE
      )
    ) +

    labs(
      x = NULL,
      y = b_y
    ) +

    theme_apex() +

    theme(
      legend.position = "none",

      panel.border = element_rect(
        colour = "black",
        fill = NA,
        linewidth = BORDER_W
      ),

      axis.text.x = element_text(
        size = B_x_text_size,
        face = "plain",
        lineheight = 0.92,
        margin = margin(t = 2)
      ),

      axis.text.y = element_text(
        size = FS_ALL,
        face = "plain"
      ),

      axis.title.y = element_text(
        size = FS_ALL,
        face = "plain",
        margin = margin(r = 4)
      ),

      # Keep B row-facet labels vertical as well
      strip.text.y = element_text(
        size = FS_ALL,
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5,
        margin = margin(
          t = 1,
          r = 2,
          b = 1,
          l = 2
        )
      ),

      strip.text.y.right = element_text(
        size = FS_ALL,
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5
      ),

      panel.spacing.y = unit(
        0.8,
        "lines"
      ),

      plot.margin = margin(
        t = 0,
        r = 0,
        b = 0,
        l = 0
      )
    )


  # =======================================================================
  # Panel C
  # =======================================================================

  pC <- ggplot() +

    geom_rect(
      data = shade_C,
      aes(
        xmin = xmin,
        xmax = xmax,
        ymin = ymin,
        ymax = ymax,
        fill = AB_level
      ),
      inherit.aes = FALSE,
      alpha = AB_FILL_ALPHA,
      colour = NA
    ) +

    annotate(
      "rect",
      xmin = 12,
      xmax = 16,
      ymin = -Inf,
      ymax = Inf,
      fill = AMP_COL,
      alpha = 0.25,
      colour = NA
    ) +

    geom_vline(
      xintercept = c(
        4,
        8,
        12,
        16
      ),
      linetype = 2,
      linewidth = VLINE_W,
      colour = "grey35",
      alpha = 0.8
    ) +

    geom_point(
      data = scalar_long,
      aes(
        x = Day,
        y = Value,
        colour = Trait
      ),
      alpha = 0.24,
      size = PT_RAW,
      stroke = 0
    ) +

    geom_errorbar(
      data = C_sum,
      aes(
        x = Day,
        ymin = ci_low,
        ymax = ci_high,
        colour = Trait
      ),
      width = 0.35,
      linewidth = LINE_ERR,
      alpha = 0.95
    ) +

    # Thinner mean trajectories
    geom_line(
      data = C_sum_line,
      aes(
        x = Day,
        y = mean,
        colour = Trait,
        linetype = Linetype,
        group = Trait
      ),
      linewidth = line_width_C
    ) +

    ggh4x::facet_grid2(
      rows = vars(
        Evo_fac_row
      ),

      cols = vars(
        AB_level
      ),

      axes = "all",
      remove_labels = "none",

      labeller = labeller(
        AB_level = ABCOL_LABELLER
      )
    ) +

    scale_fill_manual(
      values = AB_FILL_COLS,
      guide = "none"
    ) +

    scale_colour_manual(
      values = TRAIT_COLS,
      name = NULL
    ) +

    scale_linetype_identity(
      guide = "none"
    ) +

    scale_x_continuous(
      breaks = c(
        4,
        8,
        12,
        16,
        20
      ),

      labels = c(
        "4",
        "8",
        "12",
        "16",
        "20"
      ),

      limits = c(
        3,
        21
      ),

      expand = expansion(
        mult = c(
          0.002,
          0.002
        )
      )
    ) +

    scale_y_continuous(
      breaks = c(
        0,
        0.5,
        1
      ),

      labels = c(
        "0.0",
        "0.5",
        "1.0"
      ),

      limits = c(
        0,
        1
      ),

      expand = expansion(
        mult = c(
          0,
          0
        )
      )
    ) +

    coord_cartesian(
      clip = "on"
    ) +

    labs(
      x = "Day",
      y = "Community-level trait (scaled 0–1)"
    ) +

    theme_apex() +

    theme(
      legend.position = "bottom",

      legend.direction = "horizontal",
      legend.box = "horizontal",
      legend.justification = "center",

      axis.text.x = element_text(
        size = grid_tick_size,
        face = "plain",
        margin = margin(t = 1)
      ),

      axis.text.y = element_text(
        size = grid_tick_size,
        face = "plain",
        margin = margin(r = 1)
      ),

      axis.title.x = element_text(
        size = FS_ALL,
        face = "plain",
        margin = margin(t = 3)
      ),

      axis.title.y = element_text(
        size = FS_ALL,
        face = "plain",
        margin = margin(r = 4)
      ),

      strip.text.y = element_text(
        size = FS_ALL,
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5
      ),

      strip.text.y.right = element_text(
        size = FS_ALL,
        face = "plain",
        angle = -90,
        hjust = 0.5,
        vjust = 0.5
      ),

      panel.spacing.x = unit(
        0.08,
        "lines"
      ),

      panel.spacing.y = unit(
        0.08,
        "lines"
      ),

      # Bottom margin protects the shared Day title
      plot.margin = margin(
        t = 0,
        r = 0,
        b = 10,
        l = 0,
        unit = "pt"
      )
    ) +

    guides(
      colour = guide_legend(
        nrow = 1,
        byrow = TRUE,

        title.theme = element_text(
          face = "plain"
        )
      )
    )


  # =======================================================================
  # External panel-letter wrappers
  # =======================================================================

  pA_labeled <- cowplot::ggdraw() +

    cowplot::draw_plot(
      pA,
      x = 0.018,
      y = 0,
      width = 0.982,
      height = 1
    ) +

    cowplot::draw_plot_label(
      label = tags[1],
      x = tag_A_x,
      y = tag_A_y,
      size = panel_tag_size,
      fontface = "bold",
      hjust = 0,
      vjust = 1
    )


  pB_labeled <- cowplot::ggdraw() +

    cowplot::draw_plot(
      pB,
      x = 0.020,
      y = 0,
      width = 0.935,
      height = 1
    ) +

    cowplot::draw_plot_label(
      label = tags[2],
      x = tag_B_x,
      y = tag_B_y,
      size = panel_tag_size,
      fontface = "bold",
      hjust = 0,
      vjust = 1
    )


  pC_labeled <- cowplot::ggdraw() +

    cowplot::draw_plot(
      pC,
      x = 0.018,
      y = 0.030,
      width = 0.967,
      height = 0.955
    ) +

    cowplot::draw_plot_label(
      label = tags[3],
      x = tag_C_x,
      y = tag_C_y,
      size = panel_tag_size,
      fontface = "bold",
      hjust = 0,
      vjust = 1
    )


  # =======================================================================
  # Final assembly
  # =======================================================================

  # A starts slightly to the right, leaving a small blank left margin.
  # B begins at 67% and is therefore slightly wider than before.
  top_row <- cowplot::ggdraw() +

    cowplot::draw_plot(
      pA_labeled,
      x = left_pad_A,
      y = 0,
      width = rel_widths_top[1] - left_pad_A,
      height = 1
    ) +

    cowplot::draw_plot(
      pB_labeled,
      x = rel_widths_top[1],
      y = 0,
      width = rel_widths_top[2],
      height = 1
    )


  final_plot <- cowplot::ggdraw() +

    cowplot::draw_plot(
      top_row,
      x = 0,
      y = rel_heights_main[2],
      width = 1,
      height = rel_heights_main[1]
    ) +

    cowplot::draw_plot(
      pC_labeled,
      x = 0,
      y = 0,
      width = 1,
      height = rel_heights_main[2]
    )


  # -----------------------------------------------------------------------
  # Save
  # -----------------------------------------------------------------------

  grDevices::pdf(
    outfile_pdf,
    width = width_pdf,
    height = height_pdf,
    bg = "white",
    useDingbats = FALSE
  )

  print(
    final_plot
  )

  grDevices::dev.off()

  invisible(
    final_plot
  )
}


# =========================================================================
# Full Figure 4
# =========================================================================

EVO_LABELS_FULL <- c(
  "Anc. comm." =
    "Anc.\ncomm.",

  "Dom. primed" =
    "Dom.\nprimed",

  "Subdom. primed" =
    "Subdom.\nprimed",

  "Full primed" =
    "Full\nprimed"
)

obj_full <- compute_for_species(
  species = species0,
  phenos_raw = phenos_raw,

  evo_levels_keep = c(
    "Anc. comm.",
    "Dom. primed",
    "Subdom. primed",
    "Full primed"
  ),

  evo_labels_row = EVO_LABELS_FULL
)

FIG4_OUT <- file.path(
  FIG_DIR,
  "Fig4.pdf"
)

make_plots(
  obj = obj_full,
  outfile_pdf = FIG4_OUT,

  # B slightly wider; A slightly narrower
  rel_widths_top = c(
    0.67,
    0.33
  ),

  rel_heights_main = c(
    0.48,
    0.52
  ),

  # Tiny empty margin before A
  left_pad_A = 0.006,

  width_pdf = 23.5,
  height_pdf = 18.5,

  # Slightly larger A/B/C
  panel_tag_size = 30,

  tag_A_x = 0.002,
  tag_A_y = 0.845,

  tag_B_x = 0.002,
  tag_B_y = 0.990,

  tag_C_x = 0.002,
  tag_C_y = 0.985,

  # Thinner A/C trajectories
  line_width_A = LINE_MAIN * 0.6,
  line_width_C = LINE_MAIN * 0.6
)

cat(
  "Saved figure:\n  ",
  FIG4_OUT,
  "\n",
  sep = ""
)

Saved figure:
  ../figs/Fig4.pdf


### Stats

In [6]:
dist_long <- obj_full$dist_long %>%
  mutate(
    Evo_orig = factor(Evo_orig,
                      levels = c("All ancestral","Evolved 1977","Evolved 1972","All evolved")),
    AB_level = factor(AB_level, levels = c("No","Low","Intermediate","High")),
    AB_ugmL  = recode(as.character(AB_level),
                      "No" = 0, "Low" = 5, "Intermediate" = 50, "High" = 500) %>% as.numeric()
  )

df_S3 <- dist_long %>%
  filter(
    State == "Post-disturbance",
    Day == 8
  ) %>%
  distinct(Evo_orig, AB_ugmL, Replicate, distance) %>%
  rename(dist_to_postdist = distance)

m_S3 <- lm(dist_to_postdist ~ scale(AB_ugmL) * Evo_orig, data = df_S3)

cat("\nS3 | LM ANOVA: Distance to post-disturbance state (day 16 centroid), measured at day 8\n")
print(anova(m_S3))

df_S4 <- dist_long %>%
  filter(
    State == "Post-recovery",
    Day == 12
  ) %>%
  distinct(Evo_orig, AB_ugmL, Replicate, distance) %>%
  rename(dist_to_postrec = distance)

m_S4 <- lm(dist_to_postrec ~ scale(AB_ugmL) * Evo_orig, data = df_S4)

cat("\nS4 | LM ANOVA: Distance to post-recovery state (day 20 centroid), measured at day 12\n")
print(anova(m_S4))

library(emmeans)

traj <- obj_full$traj_metrics %>%
  mutate(
    Evo_orig = factor(Evo_orig,
                      levels = c("All ancestral","Evolved 1977","Evolved 1972","All evolved")),
    State = factor(State,
                   levels = c("Pre-disturbance","Post-disturbance","Post-recovery"))
  )

for (st in levels(traj$State)) {

  cat("\n============================\n")
  cat("S5 | Roughness –", st, "\n")
  cat("============================\n")

  df_st <- traj %>% filter(State == st)

  m <- lm(roughness ~ Evo_orig, data = df_st)

  cat("\nANOVA (4-level background effect):\n")
  print(anova(m))

  cat("\nPost-hoc contrasts: All pre-exposed vs each other treatment\n")

  em <- emmeans(m, ~ Evo_orig)

  contrast_results <- contrast(
    em,
    method = list(
      "All pre-exp vs All ancestral" = c(-1, 0, 0, 1),
      "All pre-exp vs Evolved 1977"  = c(0, -1, 0, 1),
      "All pre-exp vs Evolved 1972"  = c(0, 0, -1, 1)
    )
  )

  print(summary(contrast_results, infer = TRUE))
}

cat("\nMean roughness by background:\n")

means_df <- df_st %>%
  group_by(Evo_orig) %>%
  summarise(
    mean_roughness = mean(roughness),
    sd_roughness   = sd(roughness),
    .groups = "drop"
  )

print(means_df)


S3 | LM ANOVA: Distance to post-disturbance state (day 16 centroid), measured at day 8
Analysis of Variance Table

Response: dist_to_postdist
                        Df   Sum Sq  Mean Sq  F value    Pr(>F)    
scale(AB_ugmL)           1 0.194251 0.194251 174.4064 < 2.2e-16 ***
Evo_orig                 3 0.052042 0.017347  15.5750 1.745e-07 ***
scale(AB_ugmL):Evo_orig  3 0.014553 0.004851   4.3554  0.007929 ** 
Residuals               56 0.062372 0.001114                       
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

S4 | LM ANOVA: Distance to post-recovery state (day 20 centroid), measured at day 12
Analysis of Variance Table

Response: dist_to_postrec
                        Df   Sum Sq  Mean Sq F value    Pr(>F)    
scale(AB_ugmL)           1 0.064026 0.064026 51.6123 1.717e-09 ***
Evo_orig                 3 0.031828 0.010609  8.5523 9.107e-05 ***
scale(AB_ugmL):Evo_orig  3 0.004736 0.001579  1.2725    0.2927    
Residuals               56 0.069469 0.0012

### Scalar product stats

#### Computing scalar products

In [7]:
# Reconstruct 'species' table,
# including explicit control removal
species_legacy <- species_raw %>%
  mutate(
    strainID = str_replace_all(strainID, "-", "_"),
    f_correct = suppressWarnings(as.numeric(f_correct)),
    Day = parse_day(sample),
    Evo = parse_evo(sample),
    Replicate = parse_rep(sample),
    Primer_pulse = parse_ab(sample)
  ) %>%
  filter(
    !str_detect(sample, "control"),
    is.finite(f_correct),
    is.finite(Day),
    !is.na(Evo),
    is.finite(Replicate),
    is.finite(Primer_pulse)
  ) %>%
  mutate(
    Evo = factor(Evo, levels = c("All ancestral", "Evolved 1972", "Evolved 1977", "All evolved")),
    Replicate = as.integer(Replicate),
    Primer_pulse = as.numeric(Primer_pulse)
  )

# ---- IMPORTANT: lock RNG so MIC table is reproducible ----
set.seed(1)  # <- change if needed to match the SI-producing run exactly

init_pheno_legacy <- phenos_raw %>%
  mutate(
    strainID = str_replace_all(strainID, "-", "_"),
    MIC = suppressWarnings(as.numeric(MIC)),
    k   = suppressWarnings(as.numeric(k))
  ) %>%
  filter(is.finite(MIC), is.finite(k)) %>%
  select(strainID, Pop, MIC, k) %>%
  pivot_wider(
    names_from  = Pop,
    values_from = c(MIC, k),
    names_glue  = "{.value}_{Pop}"
  ) %>%
  mutate(
    noise = runif(n(), 1e-5, 1e-4)
  )

# Merge abundance rows with phenotype table
scalar_product <- species_legacy %>%
  select(strainID, f_correct, Day, Evo, Replicate, Primer_pulse) %>%
  inner_join(init_pheno_legacy, by = "strainID")

# Assign MIC/k per Evo treatment logic
scalar_anc  <- scalar_product %>% filter(Evo == "All ancestral") %>%
  mutate(MIC = MIC_ANC, k = k_ANC)

scalar_evo  <- scalar_product %>% filter(Evo == "All evolved") %>%
  mutate(MIC = MIC_EVO, k = k_EVO)

scalar_1972 <- scalar_product %>% filter(Evo == "Evolved 1972") %>%
  mutate(
    MIC = if_else(strainID == "HAMBI_1972", MIC_EVO, MIC_ANC),
    k   = if_else(strainID == "HAMBI_1972", k_EVO,   k_ANC)
  )

scalar_1977 <- scalar_product %>% filter(Evo == "Evolved 1977") %>%
  mutate(
    MIC = if_else(strainID == "HAMBI_1977", MIC_EVO, MIC_ANC),
    k   = if_else(strainID == "HAMBI_1977", k_EVO,   k_ANC)
  )

scalar_combo <- bind_rows(scalar_anc, scalar_evo, scalar_1972, scalar_1977)

# Per-strain products (MIC uses log(MIC + noise) * f_correct)
scalar_combo2 <- scalar_combo %>%
  transmute(
    strainID,
    f_correct,
    Day,
    Evo,
    Primer_pulse,
    Replicate,
    MIC,
    noise,
    k,
    scal_prod_logMIC = log(MIC + noise) * f_correct,
    scal_prod_MIC    = MIC * f_correct,
    scal_prod_k      = k * f_correct
  )

# Community-level scalar products: mean across strains within each treatment×time×rep
scalar_prods <- scalar_combo2 %>%
  group_by(Day, Evo, Primer_pulse, Replicate) %>%
  summarise(
    scal_prod_logMIC = mean(scal_prod_logMIC, na.rm = TRUE),
    scal_prod_MIC    = mean(scal_prod_MIC,    na.rm = TRUE),
    scal_prod_k      = mean(scal_prod_k,      na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    Sample = paste(Evo, Primer_pulse, Replicate, sep = "_")
  )

# GLS models (ML + AR1 within Sample + stepAIC)
M1_MIC <- gls(
  scal_prod_logMIC ~ Evo * factor(Day) * Primer_pulse,
  data = scalar_prods,
  correlation = corAR1(form = ~ 1 | Sample),
  method = "ML"
)

M2_MIC <- stepAIC(
  M1_MIC,
  scope = list(upper = ~ Evo * factor(Day) * Primer_pulse, lower = ~ 1),
  trace = FALSE
)

aov_MIC <- anova(M2_MIC)
rownames(aov_MIC) <- c(
  "(Intercept)", "Strain pre-exposure", "Day", "Pre-pulse",
  "Strain pre-exposure x day", "Strain pre-exposure x pre-pulse",
  "Day x pre-pulse", "Strain pre-exposure x day x pre-pulse"
)

#### Scalar-product GLS ANOVA

In [8]:
cat("\n============================================\n")
cat("Scalar-product GLS ANOVA | Community log MIC\n")
cat("Model: scal_prod_logMIC ~ Evo*factor(Day)*Primer_pulse with AR1 within Sample\n")
cat("============================================\n")
print(
  tibble(
    Factor  = rownames(aov_MIC),
    DF      = aov_MIC$DF,
    `F-value` = aov_MIC$`F-value`,
    `p-value` = aov_MIC$`p-value`
  ),
  n = Inf
)

M1_k <- gls(
  scal_prod_k ~ Evo * factor(Day) * Primer_pulse,
  data = scalar_prods,
  correlation = corAR1(form = ~ 1 | Sample),
  method = "ML"
)

M2_k <- stepAIC(
  M1_k,
  scope = list(upper = ~ Evo * factor(Day) * Primer_pulse, lower = ~ 1),
  trace = FALSE
)

aov_k <- anova(M2_k)
rownames(aov_k) <- c(
  "(Intercept)", "Strain pre-exposure", "Day", "Pre-pulse",
  "Strain pre-exposure x day", "Strain pre-exposure x pre-pulse",
  "Day x pre-pulse", "Strain pre-exposure x day x pre-pulse"
)

cat("\n============================================\n")
cat("Scalar-product GLS ANOVA | Community k\n")
cat("Model: scal_prod_k ~ Evo*factor(Day)*Primer_pulse with AR1 within Sample\n")
cat("============================================\n")
print(
  tibble(
    Factor  = rownames(aov_k),
    DF      = aov_k$DF,
    `F-value` = aov_k$`F-value`,
    `p-value` = aov_k$`p-value`
  ),
  n = Inf
)


Scalar-product GLS ANOVA | Community log MIC
Model: scal_prod_logMIC ~ Evo*factor(Day)*Primer_pulse with AR1 within Sample
# A tibble: 8 × 3
  Factor                                `F-value` `p-value`
  <chr>                                     <dbl>     <dbl>
1 (Intercept)                           65353.    0        
2 Strain pre-exposure                     933.    2.23e-145
3 Day                                     270.    1.02e- 94
4 Pre-pulse                               103.    7.87e- 21
5 Strain pre-exposure x day                21.4   3.87e- 33
6 Strain pre-exposure x pre-pulse           0.904 4.40e-  1
7 Day x pre-pulse                          59.5   2.71e- 36
8 Strain pre-exposure x day x pre-pulse     3.81  2.15e-  5

Scalar-product GLS ANOVA | Community k
Model: scal_prod_k ~ Evo*factor(Day)*Primer_pulse with AR1 within Sample
# A tibble: 8 × 3
  Factor                                `F-value` `p-value`
  <chr>                                     <dbl>     <dbl>
1 (Inte

### Richness and Shannon diversity stats

In [9]:
# Preprocessing
species_si <- species_raw %>%
  mutate(
    strainID     = str_replace_all(strainID, "-", "_"),
    Day          = parse_day(sample),
    Evo          = parse_evo(sample),
    Replicate    = parse_rep(sample),
    Primer_pulse = parse_ab(sample),
    f_correct    = suppressWarnings(as.numeric(f_correct)),
    count        = suppressWarnings(as.numeric(count))
  ) %>%
  filter(
    !str_detect(sample, "control"),
    is.finite(Day),
    !is.na(Evo),
    is.finite(Replicate),
    is.finite(Primer_pulse)
  ) %>%
  mutate(
    Evo         = factor(Evo, levels = c("All ancestral", "Evolved 1977", "Evolved 1972", "All evolved")),
    Replicate   = as.integer(Replicate),
    Primer_pulse = as.numeric(Primer_pulse),
    sample_time = gsub("_d.*", "", sample)
  )

# =========================
# S9 | Richness (count of strains with count > 0)
# =========================
species_pos <- species_si %>%
  filter(is.finite(count), count > 0) %>%
  mutate(present = 1)

species_pos_sums <- aggregate(
  present ~ sample * Day * Evo * Replicate * Primer_pulse,
  FUN = sum,
  data = species_pos
)

species_pos_sums$sample_time <- gsub("_d.*", "", species_pos_sums$sample)

M1_rich <- gls(
  present ~ Evo + Primer_pulse + factor(Day),
  data = species_pos_sums,
  correlation = corAR1(form = ~ 1 | sample_time),
  method = "ML"
)

M2_rich <- stepAIC(
  M1_rich,
  scope = list(upper = ~ Evo * Primer_pulse * factor(Day), lower = ~ 1),
  trace = FALSE
)

aov_rich <- as.data.frame(anova(M2_rich))
rownames(aov_rich) <- c("(Intercept)", "Strain pre-exposure", "Pre-pulse", "Day", "Pre-pulse x day")

cat("\n============================================\n")
cat("GLS ANOVA | Richness\n")
cat("Model (start): present ~ Evo + Primer_pulse + factor(Day) with AR1 within sample_time\n")
cat("============================================\n")
print(
  tibble(
    Factor    = rownames(aov_rich),
    DF        = aov_rich$DF,
    `F-value` = aov_rich$`F-value`,
    `p-value` = aov_rich$`p-value`
  ),
  n = Inf
)

# =========================
# S10 | Shannon diversity
# =========================
species_t <- dcast(species_si, sample ~ strainID, value.var = "f_correct")
rownames(species_t) <- species_t$sample
species_t <- species_t[, -1, drop = FALSE]

div_df <- as.data.frame(diversity(as.matrix(species_t), index = "shannon"))
div_df$sample <- rownames(div_df)
colnames(div_df) <- c("shannon", "sample")

meta_si <- species_si %>%
  distinct(sample, Day, Evo, Replicate, Primer_pulse) %>%
  mutate(sample_time = gsub("_d.*", "", sample))

comm_div <- merge(meta_si, div_df, by = "sample")

M1_shan <- gls(
  shannon ~ Evo + Primer_pulse + factor(Day),
  data = comm_div,
  correlation = corAR1(form = ~ 1 | sample_time),
  method = "ML"
)

M2_shan <- stepAIC(
  M1_shan,
  scope = list(upper = ~ Evo * Primer_pulse * factor(Day), lower = ~ 1),
  trace = FALSE
)

aov_shan <- as.data.frame(anova(M2_shan))
rownames(aov_shan) <- c(
  "(Intercept)", "Strain pre-exposure", "Pre-pulse", "Day",
  "Strain pre-exposure x day", "Pre-pulse x day"
)

cat("\n============================================\n")
cat("GLS ANOVA | Shannon diversity \n")
cat("Model (start): shannon ~ Evo + Primer_pulse + factor(Day) with AR1 within sample_time\n")
cat("============================================\n")
print(
  tibble(
    Factor    = rownames(aov_shan),
    DF        = aov_shan$DF,
    `F-value` = aov_shan$`F-value`,
    `p-value` = aov_shan$`p-value`
  ),
  n = Inf
)


GLS ANOVA | Richness
Model (start): present ~ Evo + Primer_pulse + factor(Day) with AR1 within sample_time
# A tibble: 5 × 3
  Factor              `F-value` `p-value`
  <chr>                   <dbl>     <dbl>
1 (Intercept)          34631.   1.10e-317
2 Strain pre-exposure      6.14 4.56e-  4
3 Pre-pulse               40.2  8.21e- 10
4 Day                    167.   8.45e- 76
5 Pre-pulse x day          3.57 7.30e-  3

GLS ANOVA | Shannon diversity 
Model (start): shannon ~ Evo + Primer_pulse + factor(Day) with AR1 within sample_time
# A tibble: 6 × 3
  Factor                    `F-value` `p-value`
  <chr>                         <dbl>     <dbl>
1 (Intercept)                 89237.   0       
2 Strain pre-exposure            26.7  2.69e-15
3 Pre-pulse                     122.   5.42e-24
4 Day                            95.9  3.73e-52
5 Strain pre-exposure x day      19.9  1.13e-31
6 Pre-pulse x day                44.1  5.03e-29
